[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toche7/SlideAdvanceDSBDI/blob/main/notebook/notebook-12-13-sql-demo.ipynb)

**Open this notebook in Google Colab**

กดปุ่มด้านบนเพื่อเปิด notebook นี้จาก GitHub เข้า Colab ได้ทันที

# Demo: SQL with SQLite on Colab
**BDAT501 Data Science Fundamentals — Module 12-13 (Afternoon)**

ใน notebook นี้เราจะใช้ `sqlite3` (built-in ใน Python) สร้างฐานข้อมูลชั่วคราวใน memory  
จากนั้นรัน SQL query และแสดงผลด้วย pandas — ไม่ต้องติดตั้ง database server ใด ๆ

## 1. Import Libraries

In [ ]:
import sqlite3
import pandas as pd

print("sqlite3 version:", sqlite3.sqlite_version)
print("pandas version:", pd.__version__)

## 2. สร้างข้อมูลตัวอย่าง (Telecom Customer Churn)

เราจำลองข้อมูลลูกค้าของบริษัทโทรคมนาคม เพื่อใช้ทดสอบ SQL query  
Schema มี 3 ตาราง: `customers`, `contracts`, `charges`

In [ ]:
# สร้างข้อมูลตัวอย่าง
customers_data = {
    "customer_id": ["C001", "C002", "C003", "C004", "C005",
                    "C006", "C007", "C008", "C009", "C010"],
    "gender": ["Male", "Female", "Female", "Male", "Female",
               "Male", "Female", "Male", "Female", "Male"],
    "senior_citizen": [0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
    "tenure_months": [12, 34, 5, 60, 24, 8, 48, 15, 3, 72],
    "churn": ["No", "No", "Yes", "No", "Yes", "Yes", "No", "No", "Yes", "No"]
}

contracts_data = {
    "customer_id": ["C001", "C002", "C003", "C004", "C005",
                    "C006", "C007", "C008", "C009", "C010"],
    "contract_type": ["Month-to-month", "One year", "Month-to-month",
                      "Two year", "Month-to-month", "Month-to-month",
                      "One year", "Two year", "Month-to-month", "Two year"],
    "payment_method": ["Electronic check", "Mailed check", "Electronic check",
                       "Bank transfer", "Electronic check", "Electronic check",
                       "Credit card", "Bank transfer", "Electronic check", "Bank transfer"]
}

charges_data = {
    "customer_id": ["C001", "C002", "C003", "C004", "C005",
                    "C006", "C007", "C008", "C009", "C010"],
    "monthly_charges": [29.85, 56.95, 53.85, 42.30, 70.70,
                        99.65, 89.10, 29.75, 104.80, 56.15],
    "total_charges": [358.2, 1889.5, 269.25, 2538.0, 1700.0,
                      797.2, 4277.0, 446.25, 315.0, 4045.0]
}

df_customers = pd.DataFrame(customers_data)
df_contracts = pd.DataFrame(contracts_data)
df_charges   = pd.DataFrame(charges_data)

print("customers:", df_customers.shape)
print("contracts:", df_contracts.shape)
print("charges:",   df_charges.shape)
df_customers.head()

## 3. โหลดข้อมูลเข้า SQLite (In-Memory)

In [ ]:
# สร้าง connection ชั่วคราวใน memory
conn = sqlite3.connect(":memory:")

# เขียน DataFrame ทั้ง 3 ตารางลง SQLite
df_customers.to_sql("customers",  conn, index=False, if_exists="replace")
df_contracts.to_sql("contracts",  conn, index=False, if_exists="replace")
df_charges.to_sql("charges",      conn, index=False, if_exists="replace")

print("โหลดข้อมูลเข้า SQLite เรียบร้อย")

# ตรวจว่ามีตารางอะไรบ้าง
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
tables

## 4. Query พื้นฐาน — SELECT, WHERE, ORDER BY

In [ ]:
query = """
SELECT customer_id, tenure_months, churn
FROM customers
WHERE churn = 'Yes'
ORDER BY tenure_months ASC;
"""

pd.read_sql_query(query, conn)

## 5. Query — GROUP BY: ลูกค้า churn แยกตาม contract_type

In [ ]:
query = """
SELECT
    c.contract_type,
    COUNT(*)                                       AS total_customers,
    SUM(CASE WHEN cu.churn = 'Yes' THEN 1 ELSE 0 END)  AS churned,
    ROUND(
        100.0 * SUM(CASE WHEN cu.churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        1
    ) AS churn_rate_pct
FROM contracts c
JOIN customers cu ON c.customer_id = cu.customer_id
GROUP BY c.contract_type
ORDER BY churn_rate_pct DESC;
"""

result = pd.read_sql_query(query, conn)
result

## 6. Query — JOIN 3 ตาราง: ค่าใช้จ่ายเฉลี่ยของลูกค้าที่ Churn vs. ไม่ Churn

In [ ]:
query = """
SELECT
    cu.churn,
    ROUND(AVG(ch.monthly_charges), 2) AS avg_monthly_charges,
    ROUND(AVG(cu.tenure_months), 1)   AS avg_tenure_months
FROM customers cu
JOIN charges ch ON cu.customer_id = ch.customer_id
GROUP BY cu.churn;
"""

pd.read_sql_query(query, conn)

## 7. (Bonus) ลองให้ LLM สร้าง SQL แล้วรันเลย

วิธีใช้:
1. เปิด ChatGPT / Gemini
2. ใส่ schema ด้านล่างนี้ลงใน prompt
3. ถามคำถามธุรกิจที่ต้องการ เช่น _"ลูกค้าที่จ่ายด้วย Electronic check มี churn rate เท่าไหร่?"_
4. เอา SQL ที่ได้มาใส่ใน cell ถัดไปแล้วรัน

```text
-- Schema
customers (customer_id TEXT, gender TEXT, senior_citizen INT,
           tenure_months INT, churn TEXT)

contracts (customer_id TEXT, contract_type TEXT, payment_method TEXT)

charges   (customer_id TEXT, monthly_charges REAL, total_charges REAL)

-- Dialect: SQLite
```

In [ ]:
# วาง SQL ที่ได้จาก LLM ที่นี่
llm_query = """
-- TODO: วาง SQL จาก ChatGPT / Gemini ที่นี่
SELECT 'วาง SQL ที่นี่' AS placeholder;
"""

# วาง SQL ที่ได้จาก LLM ที่นี่
llm_query = """
SELECT
    cu.churn,
    ROUND(AVG(ch.monthly_charges), 2) AS avg_monthly_charges,
    ROUND(AVG(cu.tenure_months), 1)   AS avg_tenure_months
FROM customers cu
JOIN charges ch ON cu.customer_id = ch.customer_id
GROUP BY cu.churn;
"""

pd.read_sql_query(llm_query, conn)

---
### หมายเหตุ
- `conn = sqlite3.connect(":memory:")` → ข้อมูลอยู่ใน RAM เท่านั้น เมื่อ session ปิดข้อมูลหายไป  
- ถ้าต้องการบันทึกถาวร ให้เปลี่ยนเป็น `sqlite3.connect("mydb.db")`  
- ทุก library (`sqlite3`, `pandas`) มีใน Colab โดยไม่ต้อง `pip install`